# Phase 1: Environment Setup
In this step, we import all necessary Python libraries.
* **Pandas & Numpy:** For data manipulation and math.
* **Seaborn & Matplotlib:** For plotting graphs.
* **Scikit-Learn:** The core library for Machine Learning models and metrics.
* **XGBoost:** An advanced gradient boosting library for high-performance modeling.
* **Joblib:** Used to save our trained models to files (`.pkl`) so the Streamlit app can use them later without retraining.

In [1]:
# Install necessary libraries (uncomment if running in Colab/Locally)
# !pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# Import Preprocessing & Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef

# Import Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Create a folder to store the saved models
os.makedirs("model", exist_ok=True)

print("✅ Libraries loaded and 'model' directory created.")

✅ Libraries loaded and 'model' directory created.


# Phase 2: Data Loading and Cleaning
Here we load the `airline_data.csv` file.
**Preprocessing Theory:**
1.  **Dropping IDs:** Columns like `id` or `Unnamed: 0` are unique identifiers for rows. They are not "features" (patterns) that help prediction, so we remove them to prevent the model from memorizing IDs.
2.  **Missing Values:** Real-world data is messy. The `Arrival Delay in Minutes` column often has missing values (NaN). We use an **Imputer** to fill these gaps with the *mean* (average) value of the column, ensuring the code doesn't crash.

In [2]:
# 1. Load Data
# Ensure your file is renamed to 'airline_data.csv' and placed in the same folder
try:
    df = pd.read_csv('airline_data.csv')
    print(f"Dataset loaded. Shape: {df.shape}")
except FileNotFoundError:
    print("❌ Error: 'airline_data.csv' not found. Please upload the file.")

# 2. Drop non-predictive columns
# IDs act as noise, not signal.
df.drop(['Unnamed: 0', 'id'], axis=1, inplace=True, errors='ignore')

# 3. Handle Missing Values
# We fill missing numerical values with the mean (average)
imputer = SimpleImputer(strategy='mean')
if 'Arrival Delay in Minutes' in df.columns:
    df['Arrival Delay in Minutes'] = imputer.fit_transform(df[['Arrival Delay in Minutes']])

print("✅ Data cleaned (IDs dropped, Missing values filled).")
print(df.head())

Dataset loaded. Shape: (103904, 25)
✅ Data cleaned (IDs dropped, Missing values filled).
   Gender      Customer Type  Age   Type of Travel     Class  Flight Distance  \
0    Male     Loyal Customer   13  Personal Travel  Eco Plus              460   
1    Male  disloyal Customer   25  Business travel  Business              235   
2  Female     Loyal Customer   26  Business travel  Business             1142   
3  Female     Loyal Customer   25  Business travel  Business              562   
4    Male     Loyal Customer   61  Business travel  Business              214   

   Inflight wifi service  Departure/Arrival time convenient  \
0                      3                                  4   
1                      3                                  2   
2                      2                                  2   
3                      2                                  5   
4                      3                                  3   

   Ease of Online booking  Gate location  ...

# Phase 3: Encoding Categorical Variables
**Theory:** Machine Learning models generally perform better with numbers, not text.
We use **Label Encoding** to convert text categories (like "Male", "Female" or "Eco", "Business") into numbers (0, 1, 2).

**Crucial Step:** We save these `LabelEncoders` into a file. Why? Because your Streamlit App needs to know that "Business Class" = 2. If we don't save the encoder, the app won't understand user inputs.

In [3]:
# Initialize a dictionary to keep track of our encoders
label_encoders = {}

# Identify categorical columns (text data)
categorical_columns = ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']

for col in categorical_columns:
    if col in df.columns:
        le = LabelEncoder()
        # Convert text to numbers
        df[col] = le.fit_transform(df[col])
        # Store the encoder
        label_encoders[col] = le

# Save the encoders for the App
joblib.dump(label_encoders, 'model/label_encoders.pkl')

print("✅ Categorical columns encoded.")
print(f"✅ Encoders saved to 'model/label_encoders.pkl'.")
print(df.head())

✅ Categorical columns encoded.
✅ Encoders saved to 'model/label_encoders.pkl'.
   Gender  Customer Type  Age  Type of Travel  Class  Flight Distance  \
0       1              0   13               1      2              460   
1       1              1   25               0      0              235   
2       0              0   26               0      0             1142   
3       0              0   25               0      0              562   
4       1              0   61               0      0              214   

   Inflight wifi service  Departure/Arrival time convenient  \
0                      3                                  4   
1                      3                                  2   
2                      2                                  2   
3                      2                                  5   
4                      3                                  3   

   Ease of Online booking  Gate location  ...  Inflight entertainment  \
0                       3     

# Phase 4: Feature Scaling and Train-Test Split
**Scaling Theory:** Algorithms like **KNN** (Distance-based) and **Logistic Regression** (Gradient-based) are sensitive to the scale of numbers. If 'Flight Distance' is 1000 and 'Age' is 30, the model might think Distance is more important just because the number is bigger. **StandardScaler** standardizes features so they have a mean of 0 and variance of 1.

**Splitting Theory:** We split data into **Training (80%)** and **Testing (20%)** sets.
* **Train:** Used to teach the model.
* **Test:** Used to evaluate performance on unseen data (simulating the real world).

In [4]:
# 1. Separate Features (X) and Target (y)
X = df.drop('satisfaction', axis=1)
y = df['satisfaction']

# 2. Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save the scaler for the App (so it can scale user inputs)
joblib.dump(scaler, 'model/scaler.pkl')

# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"✅ Data split completed.")
print(f"   - Training Samples: {X_train.shape[0]}")
print(f"   - Testing Samples: {X_test.shape[0]}")

✅ Data split completed.
   - Training Samples: 83123
   - Testing Samples: 20781


# Phase 5: Initializing the 6 Models
Here we define the **6 Mandatory Models** required by the assignment.
1.  **Logistic Regression:** A baseline linear model.
2.  **Decision Tree:** Uses if-else rules to split data.
3.  **KNN (K-Nearest Neighbors):** Classifies based on similarity to neighbors.
4.  **Naive Bayes:** Probabilistic classifier based on Bayes' theorem.
5.  **Random Forest:** An ensemble of multiple decision trees (reduces overfitting).
6.  **XGBoost:** An optimized gradient boosting framework (often the most accurate).

In [5]:
# Dictionary of models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

print("✅ All 6 models initialized.")

✅ All 6 models initialized.


# Phase 6: Training, Evaluation, and Saving
**The Loop Logic:**
For every model in our list, we:
1.  **Fit:** Train it on `X_train`.
2.  **Predict:** Test it on `X_test`.
3.  **Evaluate:** Calculate all 6 metrics (Accuracy, Precision, Recall, F1, AUC, MCC).
4.  **Save:** Export the trained model as a `.pkl` file to the `model/` folder.

This automates the entire process so you don't have to write code for each model individually.

In [6]:
results = {}

print("🚀 Starting training loop... (This may take 1-2 minutes)")

for name, model in models.items():
    print(f"   Training {name}...")

    # 1. Train
    model.fit(X_train, y_train)

    # 2. Predict
    y_pred = model.predict(X_test)

    # Calculate Probabilities (for AUC Score)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred # Fallback

    # 3. Calculate Metrics
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "AUC Score": roc_auc_score(y_test, y_prob),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }
    results[name] = metrics

    # 4. Save Model
    # Replace spaces with underscores for filename safety
    filename = f"model/{name.replace(' ', '_')}.pkl"
    joblib.dump(model, filename)

print("\n✅ All models trained, evaluated, and saved.")

🚀 Starting training loop... (This may take 1-2 minutes)
   Training Logistic Regression...
   Training Decision Tree...
   Training KNN...
   Training Naive Bayes...
   Training Random Forest...
   Training XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [19:02:19] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



✅ All models trained, evaluated, and saved.


# Phase 7: Results & Comparison
We convert our results dictionary into a **Pandas DataFrame** to create a clean comparison table.
**Assignment Requirement:** You must copy the table below into your `README.md` file under the "Comparison Table" section.

In [7]:
# Convert results to DataFrame
results_df = pd.DataFrame(results).T

# Round to 4 decimal places for readability
results_df = results_df.round(4)

print("📊 Model Performance Comparison:")
print(results_df)

# Save to CSV for easy copying
results_df.to_csv("model_metrics.csv")
print("\n✅ Metrics saved to 'model_metrics.csv'.")

📊 Model Performance Comparison:
                     Accuracy  Precision  Recall  F1 Score  AUC Score     MCC
Logistic Regression    0.8779     0.8761  0.8388    0.8570     0.9271  0.7511
Decision Tree          0.9470     0.9379  0.9408    0.9393     0.9463  0.8922
KNN                    0.9296     0.9528  0.8824    0.9163     0.9690  0.8576
Naive Bayes            0.8657     0.8645  0.8210    0.8422     0.9228  0.7262
Random Forest          0.9619     0.9742  0.9376    0.9555     0.9937  0.9228
XGBoost                0.9631     0.9723  0.9423    0.9571     0.9949  0.9252

✅ Metrics saved to 'model_metrics.csv'.
